In [1]:
import pandas as pd
import numpy as np
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error
import matplotlib.pyplot as plt
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import LabelEncoder

In [2]:
df=pd.read_csv('1.csv')

In [3]:
df  ZZZZZZZZZZZZZZ

,transaction_date,quantity,discount_applied,unit_price,product_stock,product_category,day_of_week,season,product_id
0,2020-01-01 00:00:21,7,0.04,630.03,17,Toys,Thursday,Spring,5005
1,2020-01-01 00:00:59,7,0.13,798.30,31,Electronics,Friday,Summer,1553
2,2020-01-01 00:02:01,4,0.11,858.01,80,Electronics,Sunday,Spring,3253
3,2020-01-01 00:03:11,5,0.11,463.95,66,Clothing,Wednesday,Summer,466
4,2020-01-01 00:03:22,1,0.25,118.21,4,Furniture,Tuesday,Fall,4756
...,...,...,...,...,...,...,...,...,...
999995,2021-12-31 23:55:12,7,0.08,533.49,47,Groceries,Monday,Summer,7095
999996,2021-12-31 23:55:15,1,0.05,313.86,49,Toys,Thursday,Spring,6095
999997,2021-12-31 23:56:21,8,0.04,353.45,28,Furniture,Wednesday,Fall,4893
999998,2021-12-31 23:56:51,7,0.23,666.42,78,Clothing,Monday,Summer,7509


In [4]:
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from statsmodels.tsa.statespace.sarimax import SARIMAX
import matplotlib.pyplot as plt

# Feature engineering and preprocessing function
def feature_engineering_and_preprocessing(df):
    # Convert 'transaction_date' to datetime
    df['transaction_date'] = pd.to_datetime(df['transaction_date'])

    # Extract time-based features
    df['week_of_year'] = df['transaction_date'].dt.isocalendar().week
    df['year'] = df['transaction_date'].dt.year

    # Select relevant columns for model training
    features = ['discount_applied', 'unit_price', 'product_stock', 'product_category', 'day_of_week', 'season', 'week_of_year', 'year', 'product_id']
    df = df[['transaction_date', 'quantity'] + features]

    # Define categorical and numerical features
    categorical_features = ['product_category', 'day_of_week', 'season', 'product_id']
    numerical_features = ['discount_applied', 'unit_price', 'product_stock']

    # Define the preprocessor with OneHotEncoder for categorical features and StandardScaler for numerical features
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), numerical_features),
            ('cat', OneHotEncoder(drop='first'), categorical_features)
        ]
    )

    # Fit and transform the exogenous variables
    exogenous_vars = df.drop(columns=['transaction_date', 'quantity'])
    exogenous_preprocessed = preprocessor.fit_transform(exogenous_vars)

    return df, exogenous_preprocessed, preprocessor

In [5]:
df, exogenous_preprocessed, preprocessor=feature_engineering_and_preprocessing(df)

In [6]:
df

,transaction_date,quantity,discount_applied,unit_price,product_stock,product_category,day_of_week,season,week_of_year,year,product_id
0,2020-01-01 00:00:21,7,0.04,630.03,17,Toys,Thursday,Spring,1,2020,5005
1,2020-01-01 00:00:59,7,0.13,798.30,31,Electronics,Friday,Summer,1,2020,1553
2,2020-01-01 00:02:01,4,0.11,858.01,80,Electronics,Sunday,Spring,1,2020,3253
3,2020-01-01 00:03:11,5,0.11,463.95,66,Clothing,Wednesday,Summer,1,2020,466
4,2020-01-01 00:03:22,1,0.25,118.21,4,Furniture,Tuesday,Fall,1,2020,4756
...,...,...,...,...,...,...,...,...,...,...,...
999995,2021-12-31 23:55:12,7,0.08,533.49,47,Groceries,Monday,Summer,52,2021,7095
999996,2021-12-31 23:55:15,1,0.05,313.86,49,Toys,Thursday,Spring,52,2021,6095
999997,2021-12-31 23:56:21,8,0.04,353.45,28,Furniture,Wednesday,Fall,52,2021,4893
999998,2021-12-31 23:56:51,7,0.23,666.42,78,Clothing,Monday,Summer,52,2021,7509


In [7]:
exogenous_preprocessed[0]

<1x10014 sparse matrix of type '<class 'numpy.float64'>'
	with 7 stored elements in Compressed Sparse Row format>

In [8]:
preprocessor

ColumnTransformer(transformers=[('num', StandardScaler(),
                                 ['discount_applied', 'unit_price',
                                  'product_stock']),
                                ('cat', OneHotEncoder(drop='first'),
                                 ['product_category', 'day_of_week', 'season',
                                  'product_id'])])

In [14]:
# Aggregation function to calculate weekly sales and exogenous features
def aggregate_weekly_sales(df):
    # Set 'transaction_date' as index
    df.set_index('transaction_date', inplace=True)

    # Resample the data by week and sum the sales quantity for each week
    weekly_sales = df['quantity'].resample('W').sum()

    # Resample exogenous variables and aggregate them by averaging per week
    exogenous_features = df[['discount_applied', 'unit_price', 'product_stock', 'product_category', 'day_of_week', 'season', 'product_id']]
    exogenous_weekly = exogenous_features.resample('W').mean()

    return weekly_sales, exogenous_weekly

In [15]:
weekly_sales, exogenous_weekly=aggregate_weekly_sales(df)

KeyError: "None of ['transaction_date'] are in the columns"

In [16]:
weekly_sales

transaction_date
2020-01-05    34262
2020-01-12    48801
2020-01-19    48670
2020-01-26    47417
2020-02-02    47424
              ...  
2021-12-05    49003
2021-12-12    48490
2021-12-19    47796
2021-12-26    47716
2022-01-02    34962
Freq: W-SUN, Name: quantity, Length: 105, dtype: int64

In [17]:
exogenous_weekly

,discount_applied,unit_price,product_stock,product_id
transaction_date,,,,
2020-01-05,0.247444,493.842039,49.547090,5024.245393
2020-01-12,0.250218,500.360408,49.237417,4994.892471
2020-01-19,0.253845,501.173528,49.123711,4984.386804
2020-01-26,0.248498,500.717594,49.704515,5023.779721
2020-02-02,0.248451,500.971152,49.768835,4972.446885
...,...,...,...,...
2021-12-05,0.250253,501.404363,49.113029,4981.348335
2021-12-12,0.250145,501.238421,49.634871,4974.941293
2021-12-19,0.251581,502.487071,49.773011,4936.150146


In [18]:

# Model training and forecasting function
def train_sarimax_model(df, product_id, order=(1, 1, 1), seasonal_order=(1, 1, 1, 52), forecast_steps=12):
    # Filter the dataset for the specific product ID
    df_product = df[df['product_id'] == product_id]

    # Apply feature engineering and preprocessing
    df_product, exogenous_preprocessed, preprocessor = feature_engineering_and_preprocessing(df_product)

    # Aggregate weekly sales and exogenous features for modeling
    weekly_sales, exogenous_weekly = aggregate_weekly_sales(df_product)

    # Train the SARIMAX model using weekly sales and exogenous variables
    model = SARIMAX(weekly_sales, exog=exogenous_weekly, order=order, seasonal_order=seasonal_order)
    model_fit = model.fit()

    # Forecast sales for the next 'forecast_steps' weeks
    forecast = model_fit.forecast(steps=forecast_steps, exog=exogenous_weekly[-forecast_steps:])

    # Plot historical and forecasted sales
    plt.figure(figsize=(10,6))
    plt.plot(weekly_sales.index, weekly_sales, label='Historical Sales')
    plt.plot(pd.date_range(weekly_sales.index[-1], periods=forecast_steps+1, freq='W')[1:], forecast, color='red', label='Forecasted Sales')
    plt.title(f'Sales Forecast for Product ID {product_id} - Next {forecast_steps} Weeks')
    plt.xlabel('Date')
    plt.ylabel('Quantity Sold')
    plt.legend()
    plt.show()

    return forecast

In [19]:
# Train and forecast for a specific product (example product_id = 5005)
forecast = train_sarimax_model(df, product_id=5005, order=(1,1,1), seasonal_order=(1,1,1,52), forecast_steps=12)

KeyError: 'transaction_date'